# n5 · 神经网络模块与优化器  Module, Linear & SGD

> **能力标签**：SH8（PyTorch 基础 / PyTorch fundamentals）

## 目标 Objectives

完成本课后，你应该能够：

1. 解释 **Module**（模块）抽象的设计意图：统一管理参数、零梯度与前向传播。
2. 实现 **Linear**（线性层）：`y = x @ W + b`，并通过 `parameters()` 暴露 `W` 和 `b`。
3. 实现 **SGD** 优化器：`zero_grad()` 与 `step()`。
4. 理解**均值损失**（mean loss）与 `mean()` 反向传播：梯度均匀分配到每个元素，即 `grad / n`。
5. 用以上组件组装一个完整的**训练循环**（training loop），在合成回归数据集上拟合线性模型。

> 对应能力：**SH8**

## 概念讲解 Concepts

### 1. Module 抽象 Module Abstraction

PyTorch 的 `nn.Module` 提供了神经网络层的统一接口。我们仿照它实现一个极简版本：

```python
class Module:
    def parameters(self):
        return []                      # 子类覆盖，返回 Tensor 列表

    def zero_grad(self):
        for p in self.parameters():
            p.grad = np.zeros_like(p.data)   # 清零所有参数梯度
```

**为何需要 Module？**：当网络层增多时，手动管理每个参数会非常繁琐。`Module.parameters()` 提供统一的参数列表接口，`zero_grad()` 一次清零所有梯度。

---

### 2. Linear 层 Linear Layer

$$y = x @ W + b$$

- 输入 $x$：形状 `(B, in_features)`，B 为批大小（batch size）
- 权重 $W$：形状 `(in_features, out_features)`，Kaiming 初始化：$W \sim \mathcal{N}(0, 1/\sqrt{\text{in\_features}})$
- 偏置 $b$：形状 `(out_features,)`，初始为 0

矩阵乘法广播自动处理批维度，偏置 `b` 通过广播加到每个样本。

---

### 3. SGD 优化器 SGD Optimizer

**随机梯度下降**（Stochastic Gradient Descent）更新规则：

$$\theta_{t+1} = \theta_t - \eta \cdot \nabla_{\theta} L$$

```python
class SGD:
    def __init__(self, params, lr=0.1):
        self.params = list(params)
        self.lr = lr

    def zero_grad(self):               # 清零梯度（与 Module.zero_grad 等价）
        for p in self.params:
            p.grad = np.zeros_like(p.data)

    def step(self):                    # 参数更新
        for p in self.params:
            p.data = p.data - self.lr * p.grad
```

---

### 4. 均值损失与 `mean()` 反向传播

**均方误差**（MSE）通常用均值而非求和，以便学习率对批大小不敏感：

$$L = \frac{1}{n} \sum_{i=1}^n \ell_i$$

`mean()` 的反向传播：$\partial L / \partial \ell_i = 1/n$，即梯度均匀分配给所有元素：

```python
def mean(self):
    n = self.data.size
    out = Tensor(self.data.mean(), (self,), "mean")
    def _backward():
        self.grad = self.grad + np.ones_like(self.data) * out.grad / n
    out._backward = _backward
    return out
```

---

### 5. 完整训练循环 Full Training Loop

```
for epoch in range(epochs):
    # 1. 前向传播 Forward pass
    y_pred = model(X_train)
    loss = mse(y_pred, y_train)

    # 2. 清零梯度 Zero gradients  ← 每步必须！
    optimizer.zero_grad()

    # 3. 反向传播 Backward pass
    loss.backward()

    # 4. 参数更新 Update parameters
    optimizer.step()
```

这与 PyTorch 的标准训练循环**完全一致**。

## 示例 Worked Example

In [ ]:
# ── nanograd-l5 完整实现（Tensor + Module + Linear + SGD）──────────────────
import numpy as np


def _unbroadcast(grad, shape):
    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)
    for i, s in enumerate(shape):
        if s == 1 and grad.shape[i] != 1:
            grad = grad.sum(axis=i, keepdims=True)
    return grad


class Tensor:
    def __init__(self, data, _children=(), _op=""):
        self.data = np.asarray(data, dtype=float)
        self.grad = np.zeros_like(self.data)
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __add__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data + other.data, (self, other), "+")
        def _backward():
            self.grad = self.grad + _unbroadcast(out.grad, self.data.shape)
            other.grad = other.grad + _unbroadcast(out.grad, other.data.shape)
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data * other.data, (self, other), "*")
        def _backward():
            self.grad = self.grad + _unbroadcast(other.data * out.grad, self.data.shape)
            other.grad = other.grad + _unbroadcast(self.data * out.grad, other.data.shape)
        out._backward = _backward
        return out

    def __matmul__(self, other):
        out = Tensor(self.data @ other.data, (self, other), "@")
        def _backward():
            self.grad = self.grad + out.grad @ other.data.T
            other.grad = other.grad + self.data.T @ out.grad
        out._backward = _backward
        return out

    def sum(self):
        out = Tensor(self.data.sum(), (self,), "sum")
        def _backward():
            self.grad = self.grad + np.ones_like(self.data) * out.grad
        out._backward = _backward
        return out

    def mean(self):
        """均值（mean）：梯度均匀分配给 n 个元素 (grad / n)."""
        n = self.data.size
        out = Tensor(self.data.mean(), (self,), "mean")
        def _backward():
            self.grad = self.grad + np.ones_like(self.data) * out.grad / n
        out._backward = _backward
        return out

    def relu(self):
        out = Tensor(np.maximum(0.0, self.data), (self,), "relu")
        def _backward():
            self.grad = self.grad + (self.data > 0) * out.grad
        out._backward = _backward
        return out

    def __neg__(self): return self * -1.0

    def backward(self):
        topo, visited = [], set()
        def build(v):
            if id(v) not in visited:
                visited.add(id(v))
                for child in v._prev: build(child)
                topo.append(v)
        build(self)
        self.grad = np.ones_like(self.data)
        for v in reversed(topo): v._backward()

    def __repr__(self):
        return f"Tensor(shape={self.data.shape})"


class Module:
    """参数容器基类（Base class for parameter containers)."""
    def parameters(self): return []
    def zero_grad(self):
        for p in self.parameters():
            p.grad = np.zeros_like(p.data)


class Linear(Module):
    """线性层（Linear layer）: y = x @ W + b."""
    def __init__(self, nin, nout, seed=0):
        rng = np.random.default_rng(seed)
        scale = 1.0 / np.sqrt(nin)           # Kaiming 初始化
        self.w = Tensor(rng.normal(0, scale, (nin, nout)))
        self.b = Tensor(np.zeros(nout))

    def __call__(self, x):
        return x @ self.w + self.b

    def parameters(self):
        return [self.w, self.b]


class SGD:
    """随机梯度下降（Stochastic Gradient Descent）优化器."""
    def __init__(self, params, lr=0.1):
        self.params = list(params)
        self.lr = lr

    def zero_grad(self):
        for p in self.params:
            p.grad = np.zeros_like(p.data)

    def step(self):
        for p in self.params:
            p.data = p.data - self.lr * p.grad


print("Module / Linear / SGD 定义完毕")


In [ ]:
# ── 参数管理演示 Parameter Management Demo ──────────────────────────────────
linear = Linear(4, 3, seed=0)

params = linear.parameters()
print("Linear(4, 3) 参数列表:")
for i, p in enumerate(params):
    print(f"  [{i}] shape={p.data.shape},  name={'W' if i==0 else 'b'}")

total = sum(p.data.size for p in params)
print(f"  总参数量 (Total params): {total}  (期望: 4×3 + 3 = {4*3+3})")

# 零梯度验证
linear.zero_grad()
for p in params:
    assert np.all(p.grad == 0), "梯度未清零！"
print("✓ zero_grad() 正常工作")

# 前向传播形状验证
X = Tensor(np.random.randn(5, 4))   # 批大小 5，输入维度 4
y = linear(X)
print(f"\n前向传播 Forward pass:")
print(f"  X.shape={X.data.shape} → y.shape={y.data.shape}  (期望 (5,3))")
assert y.data.shape == (5, 3)
print("✓ 输出形状正确")


In [ ]:
# ── 完整训练循环：在合成线性数据上拟合 Full Training Loop ───────────────────
# 合成数据：y = X @ W_true + noise，任务是用 Linear 恢复近似的 W_true

np.random.seed(2025)
N, D_in, D_out = 64, 8, 4
W_true = np.random.randn(D_in, D_out)
X_np = np.random.randn(N, D_in)
Y_np = X_np @ W_true + 0.01 * np.random.randn(N, D_out)

X_train = Tensor(X_np)
Y_train = Tensor(Y_np)

# 模型与优化器
model = Linear(D_in, D_out, seed=0)
optimizer = SGD(model.parameters(), lr=0.05)

EPOCHS = 200
losses = []

for epoch in range(EPOCHS):
    # 1. 前向传播
    Y_pred = model(X_train)
    diff = Y_pred + (Y_train * -1.0)   # (Y_pred - Y_train)
    loss = (diff * diff).mean()         # MSE 均值损失

    # 2. 清零梯度
    optimizer.zero_grad()

    # 3. 反向传播
    loss.backward()

    # 4. 更新参数
    optimizer.step()

    losses.append(float(loss.data))
    if epoch % 50 == 0 or epoch == EPOCHS - 1:
        print(f"  epoch {epoch:3d}  MSE loss = {loss.data:.6f}")

print()
print(f"初始损失 Initial loss : {losses[0]:.6f}")
print(f"最终损失 Final loss   : {losses[-1]:.6f}")
assert losses[-1] < losses[0] * 0.1, "损失未下降至 10% 以下，训练失败！"
print("✓ 训练收敛，损失下降超过 90%")

# 验证学习到的权重接近真实权重
W_learned = model.w.data
corr = np.corrcoef(W_true.ravel(), W_learned.ravel())[0, 1]
print(f"\n学习权重与真实权重的相关系数（Pearson r）= {corr:.4f}  (期望 > 0.99)")
assert corr > 0.99, f"权重相关系数太低: {corr:.4f}"
print("✓ 权重估计准确 Weight estimation accurate")


## 动手 Exercise

In [ ]:
# ── 动手练习：双层 MLP 训练 Two-Layer MLP Training ──────────────────────────
# 场景：二分类问题（非线性分离的合成数据）
# 模型：Linear(2, 8) → ReLU → Linear(8, 1) → 输出标量

np.random.seed(42)
N = 200

# 合成同心圆数据
r = np.random.uniform(0, 1, N)
theta = np.random.uniform(0, 2 * np.pi, N)
X_cls_np = np.stack([r * np.cos(theta), r * np.sin(theta)], axis=1)
Y_cls_np = (r > 0.5).astype(float).reshape(-1, 1) * 2 - 1  # ±1 标签


class TwoLayerMLP(Module):
    def __init__(self, seed=0):
        self.l1 = Linear(2, 8, seed=seed)
        self.l2 = Linear(8, 1, seed=seed + 1)

    def __call__(self, x):
        h = self.l1(x).relu()
        return self.l2(h)

    def parameters(self):
        return self.l1.parameters() + self.l2.parameters()


mlp = TwoLayerMLP(seed=0)
optimizer = SGD(mlp.parameters(), lr=0.05)

X_cls = Tensor(X_cls_np)
Y_cls = Tensor(Y_cls_np)

print(f"模型参数量: {sum(p.data.size for p in mlp.parameters())}")
print(f"  期望: 2×8+8 + 8×1+1 = {2*8+8 + 8+1}")

EPOCHS = 150
for epoch in range(EPOCHS):
    y_pred = mlp(X_cls)
    diff = y_pred + (Y_cls * -1.0)
    loss = (diff * diff).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0 or epoch == EPOCHS - 1:
        # 计算准确率
        preds = (y_pred.data.ravel() > 0).astype(int)
        labels = (Y_cls_np.ravel() > 0).astype(int)
        acc = (preds == labels).mean()
        print(f"  epoch {epoch:3d}  loss={loss.data:.4f}  acc={acc:.3f}")

print()
acc_final = ((mlp(X_cls).data.ravel() > 0) == (Y_cls_np.ravel() > 0)).mean()
print(f"最终准确率 Final accuracy: {acc_final:.3f}  (期望 > 0.75)")
assert acc_final > 0.75, f"准确率不足: {acc_final:.3f}"
print("✓ 双层 MLP 训练成功")


## 延伸阅读 Further Reading

1. **PyTorch `nn.Module` 文档**：<https://pytorch.org/docs/stable/generated/torch.nn.Module.html> — 工业级模块抽象，与本课 `Module` 类直接对应。
2. **PyTorch `nn.Linear` 源码**：<https://github.com/pytorch/pytorch/blob/main/torch/nn/modules/linear.py> — 可与本课 `Linear` 对比，看工业版本添加了哪些功能。
3. **Ruder (2016) «Overview of Gradient Descent»**：<https://ruder.io/optimizing-gradient-descent/> — 从 SGD 到 Adam、RMSprop 的全面综述。
4. **Goodfellow et al. «Deep Learning»** 第 8 章（优化）：学习率调度、动量、自适应方法的理论基础。
5. **CS231n Assignment 2**：<https://cs231n.github.io/assignments2024/assignment2/> — 手写 Linear、BatchNorm、Dropout 的经典练习。

## 关联练习 Related Assignments

```bash
python check.py 02-nanograd-l5
```

> 实现 `Module`、`Linear`、`SGD` 及 `Tensor.mean()`，组装训练循环在合成数据上拟合线性层。

> 能力标签：**SH8** · threshold ≥ 0.7